## Conducción del calor:
Se consideran dos conductores térmicos $C_i$, $i=1,2$ envueltos en un recinto $C_0$. El elemento $C_1$ está a  temperatura constante $u_1=60^o$, mientras que el otro tiene una conductividad térmica quintuple de la $C_0$. Se suponge que la frontera exterior de $C_0$ está a $20^oC$ y que el sistema está en equilibrio térmico. La temperatura en el recinto vendrá dada por la solución de
$$\nabla \cdot (\kappa\nabla u) = 0\text{ en }\Omega,$$
con la condición frontera
$$\left. u\right|_\Gamma=g,$$
siendo $\Omega$ el interior de $C_0$ menos $C_1$, $\Gamma = \partial \Omega$ y $g$ una función tal que $u=u_i$ en $C_i$.

<img src="graphs/conductor.svg">

### Importamos módulos

In [1]:
%reset -f
import mfem.ser as mfem
from glvis import glvis # visualización

### Lectura de malla

In [2]:
mesh = mfem.Mesh('mallas/conductor.mesh')

In [3]:
glvis(mesh,keys="bpppl")

In [4]:
print("Etiquetas en la frontera:",mesh.bdr_attributes.ToList())
print("Etiquetas interiores:",mesh.attributes.ToList())

Etiquetas en la frontera: [2, 99, 100]
Etiquetas interiores: [1, 5]


### Espacio de elementos finitos

In [5]:
fec = mfem.H1_FECollection(1,  mesh.Dimension())
fespace = mfem.FiniteElementSpace(mesh, fec)

### Formulación variacional
Hallar $u\in g+ H^1_0(\Omega)$ tal que
$$ \int_\Omega \kappa(x)\nabla u \cdot \nabla v\,dx = 0,\quad \forall v \in H^1_0(\Omega).$$

### Coeficiente $\kappa$

#### Coeficiente $\kappa$ definido por localización de los puntos del dominio

Una forma de definir un coeficiente definido a trozos: dando una función (PyCoefficient) en la que se evalúa cada trozo mediante la localización de los puntos

In [ ]:
# Coeficiente k
class kappaCoef(mfem.PyCoefficient):
    def EvalValue(self, x):
        return float(1 + 4*(x[0]<-1)*(x[0]>-2)*(x[1]<3)*(x[1]>-3))

# Coeficiente a trozos
kappa = kappaCoef()

In [ ]:
# Formas bilineal y lineal
a = mfem.BilinearForm(fespace)
a.AddDomainIntegrator(mfem.DiffusionIntegrator(kappa))
a.Assemble()

# Segundo miembro (f(x)=0)
b = mfem.LinearForm(fespace)
b.Assign(0.)

#### Coeficiente $\kappa$ definido por proyección sobre etiquetas de la malla

Una forma alternativa de definir un coeficiente a trozos consiste en usar el método `RestrictedCoefficient`, como sigue:
- debemos encontrar los nodos interiores (`mesh.attributes`) de la malla en los que vamos a dar el valor de una función. 
- para ello, definimos una lista de ceros, de longitud, el valor de la mayor etiqueta de la malla, y colocamos un 1 en el elemento de esa lista que corresponde con la etiqueta que vamos a activar.
Por ejemplo, la malla `conductor.mesh` contiene las etiquetas interiores 1 y 5. La etiqueta 1 corresponde a la etiqueta del rectángulo interior donde el conductor tiene conductividad quintuple de la del resto (que tiene etiqueta 5)

In [ ]:
# etiquetas de los nodos para imponer la función a trozos

# definimos lista de etiquetas nulas
ess_list = [0]*mesh.attributes.Max()
# marcamos un 1 en la quinta etiqueta (todo el conductor salvo el rectángulo interior)
ess_list[4]=1
print(ess_list)
etiq1 = mfem.intArray(ess_list)

# Función a imponer sobre esos nodos ()
one = mfem.ConstantCoefficient(1.0)
kappa1 = mfem.RestrictedCoefficient(one,etiq1)

## Repetimos para el otro trozo

# definimos lista de etiquetas nulas
ess_list = [0]*mesh.attributes.Max()
# marcamos un 1 en la primera etiqueta (rectángulo interior)
ess_list[0]=1
print(ess_list)
etiq2 = mfem.intArray(ess_list)

two = mfem.ConstantCoefficient(5.0)
kappa2 = mfem.RestrictedCoefficient(two,etiq2)

In [ ]:
# Forma bilineal
a = mfem.BilinearForm(fespace)

a.AddDomainIntegrator(mfem.DiffusionIntegrator(kappa1))
a.AddDomainIntegrator(mfem.DiffusionIntegrator(kappa2))
a.Assemble()

# Segundo miembro (f(x)=0)
b = mfem.LinearForm(fespace)
b.Assign(0.)

#### Marcando la etiqueta sobre `AddDomainIntegrator` 

In [ ]:
ess_list = [0]*mesh.attributes.Max()
# marcamos un 1 en la quinta etiqueta (todo el conductor salvo el rectángulo interior)
ess_list[4]=1
etiq1 = mfem.intArray(ess_list)
ess_list = [0]*mesh.attributes.Max()
# marcamos un 1 en la primera etiqueta (rectángulo interior)
ess_list[0]=1
etiq2 = mfem.intArray(ess_list)


# Forma bilineal
a = mfem.BilinearForm(fespace)

a.AddDomainIntegrator(mfem.DiffusionIntegrator(mfem.ConstantCoefficient(1.)), etiq1)
a.AddDomainIntegrator(mfem.DiffusionIntegrator(mfem.ConstantCoefficient(5.)), etiq2)
a.Assemble()

# Segundo miembro (f(x)=0)
b = mfem.LinearForm(fespace)
b.Assign(0.)

#### Coeficiente $\kappa$ definido mediante `PWConstCoefficient`

In [6]:
ess_list = [0.]*mesh.attributes.Max()
ess_list[4] = 1.
ess_list[0] = 5.
ess = mfem.Vector(ess_list)
kappa = mfem.PWConstCoefficient(ess)

In [7]:
# Formas bilineal y lineal
a = mfem.BilinearForm(fespace)
a.AddDomainIntegrator(mfem.DiffusionIntegrator(kappa))
a.Assemble()

# Segundo miembro (f(x)=0)
b = mfem.LinearForm(fespace)
b.Assign(0.)

<mfem._ser.linearform.LinearForm; proxy of <Swig Object of type '_p_mfem__LinearForm' at 0x7fe643f864b0> >

### Condición Dirichlet

En este caso, las etiquetas frontera de la malla son <code>[2, 99, 100]</code> con 2 para $C_0$, 99 para $C_1$ y 100 para $C_2$.
Entonces debemos definir una función que valga 20 en la etiqueta de $C_0$ y 60 en la de $C_1$.

##### Separadamente con `ProjectBdrCoefficient`: 

In [8]:
x = mfem.GridFunction(fespace)
x.Assign(0.0)

ess_tdof_list = mfem.intArray()
etiquetas = mfem.intArray()
# definimos lista de etiquetas nulas
ess_list = [0]*mesh.bdr_attributes.Max()
# marcamos un 1 en la segunda 2 (etiqueta C0)
ess_list[1]=1
ess_bdr = mfem.intArray(ess_list)

# valor 20 en C0        
f0 = mfem.ConstantCoefficient(20.0)
x.ProjectBdrCoefficient(f0,ess_bdr)
# recopilamos etiquetas para pasarlas al solver
fespace.GetEssentialTrueDofs(ess_bdr, etiquetas)
ess_tdof_list.Append(etiquetas)

# Repetimos para C1

# definimos lista de etiquetas nulas
ess_list = [0]*mesh.bdr_attributes.Max()
# marcamos un 1 en la posicion 100
ess_list[99]=1
ess_bdr = mfem.intArray(ess_list)

f1 = mfem.ConstantCoefficient(60.0)
x.ProjectBdrCoefficient(f1,ess_bdr)

# Recopilamos todas las etiquetas
fespace.GetEssentialTrueDofs(ess_bdr, etiquetas)

ess_tdof_list.Append(etiquetas)

100

##### Toda a la vez con `PWConstCoefficient`

In [9]:
ess_list = [0]*mesh.bdr_attributes.Max()
ess_list[1] = 20.
ess_list[99] = 60.
ess = mfem.Vector(ess_list)

border = mfem.PWConstCoefficient(ess)

ess_lt = [0]*mesh.bdr_attributes.Max()
ess_lt[1] = 1
ess_lt[99] = 1
essl = mfem.intArray(ess_lt)
x = mfem.GridFunction(fespace)
x.Assign(0.)
x.ProjectBdrCoefficient(border,essl)

ess_tdof_list = mfem.intArray()
fespace.GetEssentialTrueDofs(essl, ess_tdof_list)

<div class="alert-warning">
Nótese que aunque toda la frontera es Dirichlet, no podemos usar el método `GetBoundaryTrueDofs` pues $C_2$ tiene etiquetas en la frontera asignadas, aunque sean nodos internos.
</div>

### Formulación del sistema

In [10]:
A = mfem.SparseMatrix()
B = mfem.Vector()
X = mfem.Vector()
a.FormLinearSystem(ess_tdof_list, x, b, A, X, B)
print("Tamaño del sistema lineal: " + str(A.Height()))

Tamaño del sistema lineal: 604


###  Resolución del sistema

In [11]:
# Precondicionador tipo Gauss-Seidel
M = mfem.GSSmoother(A)
mfem.PCG(A, M, B, X, 0, 200, 1e-12, 0.0)

# Asignamos solución a la función grid
a.RecoverFEMSolution(X, b, x)

### Visualización

In [12]:
glvis((mesh, x),keys='R*****') 